# CordiaAI — Step 16: Echo branch (EchoNet-Dynamic + ViT-B/16 per frame + temporal transformer)

Predicts **left ventricular ejection fraction (EF, %)** from an apical 4-chamber echo video. Each sampled
frame becomes a ViT-B/16 token; a 2-layer temporal transformer reads the 16 frames (one heartbeat) and
regresses EF. This is the methodology's "echo branch", and EF is the single most important input for the
digital twin's contractility.

**Before running:**
1. Register on Stanford AIMI's portal and accept the EchoNet-Dynamic Research Use Agreement
   (`python download_imaging_datasets.py info echonet_dynamic`), then put `EchoNet-Dynamic.zip` in your
   Google Drive at `MyDrive/cordia/EchoNet-Dynamic.zip`. The agreement forbids redistribution, so keep it
   in your own Drive only.
2. `Runtime → Change runtime type → T4 GPU`.

**Split:** EchoNet-Dynamic's own `Split` column (TRAIN / VAL / TEST).

**Same code as the app:** the cells marked `%%writefile` are copies of this project's own modules
(`src/models/imaging/`, `src/imaging/`), generated from them, so the model trained here is exactly
the one `mobile_app/backend/main.py` loads.

In [ ]:
!pip install -q transformers opencv-python-headless
import torch
print('CUDA:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
import os
for d in ['src/models/imaging', 'src/imaging']:
    os.makedirs(d, exist_ok=True)
for f in ['src/__init__.py', 'src/models/__init__.py', 'src/models/imaging/__init__.py', 'src/imaging/__init__.py']:
    open(f, 'a').close()

In [ ]:
%%writefile src/models/imaging/vit_backbone.py
"""Shared ViT-B/16 backbone for the imaging branches (X-ray, echo).

Same pretrained model the ECG branch uses (google/vit-base-patch16-224-in21k),
so every image-like branch in CordiaAI is a Vision Transformer.

Two details worth knowing:
- `pretrained=False` builds the same architecture with random weights. Local
  inference uses this, since the trained checkpoint overwrites every weight
  anyway -- it avoids downloading ~350MB of ImageNet weights at app startup.
- Layer freezing matches parameter names with a regex that accepts both
  "encoder.layer.N." and "layers.N.", since transformers versions differ
  (see the note in notebooks/step5_ecg_vit_colab.ipynb).
"""
from __future__ import annotations

import re

from torch import nn
from transformers import ViTConfig, ViTModel

VIT_NAME = "google/vit-base-patch16-224-in21k"
_LAYER_RE = re.compile(r"\blayers?\.(\d+)\.")


def load_vit(pretrained: bool = True, config: ViTConfig | None = None) -> ViTModel:
    if config is not None:
        return ViTModel(config)
    return ViTModel.from_pretrained(VIT_NAME) if pretrained else ViTModel(ViTConfig())


def freeze_below(vit: ViTModel, train_from_layer: int) -> None:
    """Train only encoder layers >= train_from_layer (plus nothing else in the ViT)."""
    for name, p in vit.named_parameters():
        match = _LAYER_RE.search(name)
        p.requires_grad = bool(match) and int(match.group(1)) >= train_from_layer


class Projection(nn.Sequential):
    def __init__(self, d_in: int, d_embed: int):
        super().__init__(nn.Linear(d_in, d_embed), nn.LayerNorm(d_embed))

In [ ]:
%%writefile src/models/imaging/echo_vit.py
"""Echocardiogram branch: ejection fraction from an echo video.

Each frame goes through the shared ViT-B/16 and becomes one 256-dim token;
a small temporal transformer then reads the sequence of frame tokens (the
heart contracting and relaxing) and regresses ejection fraction (EF, %).
This "image transformer per frame + transformer over time" design is the
factorised form of a video transformer, and keeps memory low enough for a
free Colab T4 by freezing the lower ViT layers.

Trained on EchoNet-Dynamic in `notebooks/step16_echo_ef_colab.ipynb`.
"""
from __future__ import annotations

import torch
from torch import nn
from transformers import ViTConfig

from src.models.imaging.vit_backbone import Projection, freeze_below, load_vit

N_FRAMES = 16  # frames sampled per clip; EchoNet-Dynamic clips are ~50 fps, so 16 frames at stride 2 ~ one beat


class EchoViT(nn.Module):
    def __init__(
        self,
        n_frames: int = N_FRAMES,
        d_embed: int = 256,
        train_from_layer: int = 10,
        temporal_depth: int = 2,
        heads: int = 8,
        dropout: float = 0.1,
        pretrained: bool = True,
        config: ViTConfig | None = None,
    ):
        super().__init__()
        self.vit = load_vit(pretrained, config)
        freeze_below(self.vit, train_from_layer)
        self.frame_project = Projection(self.vit.config.hidden_size, d_embed)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_embed))
        self.time_embed = nn.Parameter(torch.zeros(1, n_frames + 1, d_embed))
        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.time_embed, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d_embed, nhead=heads, dim_feedforward=d_embed * 4, dropout=dropout,
            activation="gelu", batch_first=True,
        )
        self.temporal = nn.TransformerEncoder(layer, num_layers=temporal_depth)
        self.norm = nn.LayerNorm(d_embed)
        self.ef_head = nn.Linear(d_embed, 1)

    def embed(self, clips: torch.Tensor) -> torch.Tensor:
        """clips: (batch, frames, 3, 224, 224) -> E_e, the 256-dim echo embedding."""
        b, t = clips.shape[:2]
        frames = self.vit(pixel_values=clips.flatten(0, 1)).last_hidden_state[:, 0]
        tokens = self.frame_project(frames).view(b, t, -1)
        seq = torch.cat([self.cls_token.expand(b, -1, -1), tokens], dim=1) + self.time_embed[:, : t + 1]
        return self.norm(self.temporal(seq)[:, 0])

    def forward(self, clips: torch.Tensor) -> torch.Tensor:
        """Returns predicted EF in percent, shape (batch,)."""
        return self.ef_head(self.embed(clips)).squeeze(-1)

In [ ]:
%%writefile src/imaging/preprocess.py
"""Image preprocessing shared by training notebooks and the app.

Keeping one copy of each conversion matters for the same reason it did for
the ECG spectrogram (see gradcam_ecg.py): the app must feed a model exactly
what it was trained on, or predictions silently degrade.
"""
from __future__ import annotations

import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

VIT_SIZE = 224


def _vit_normalise(x: np.ndarray) -> np.ndarray:
    # google/vit-base-patch16-224-in21k's preprocessor: scale to [0,1], then mean 0.5 / std 0.5
    return (x.astype(np.float32) / 255.0 - 0.5) / 0.5


# ---------------------------------------------------------------- X-ray
def xray_to_tensor(image: Image.Image) -> torch.Tensor:
    """Any chest X-ray image (PNG/JPG, grey or colour) -> (3, 224, 224)."""
    grey = image.convert("L").resize((VIT_SIZE, VIT_SIZE), Image.BILINEAR)
    arr = np.stack([np.array(grey)] * 3, axis=-1)
    return torch.from_numpy(_vit_normalise(arr)).permute(2, 0, 1)


# ---------------------------------------------------------------- Echo
def read_video_frames(path: str | Path) -> np.ndarray:
    """All frames of a video file as (frames, H, W) uint8 grey."""
    import cv2

    cap = cv2.VideoCapture(str(path))
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    cap.release()
    if not frames:
        raise ValueError(f"No frames could be read from {path}")
    return np.stack(frames)


def read_video_bytes(data: bytes, suffix: str = ".avi") -> np.ndarray:
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / f"upload{suffix}"
        path.write_bytes(data)
        return read_video_frames(path)


def sample_clip(frames: np.ndarray, n_frames: int, stride: int = 2, start: int | None = None) -> np.ndarray:
    """n_frames frames, `stride` apart, starting at `start` (default: centred).
    Short videos are looped, as in the EchoNet-Dynamic reference code."""
    span = n_frames * stride
    if start is None:
        start = max(0, (len(frames) - span) // 2)
    idx = (start + np.arange(n_frames) * stride) % len(frames)
    return frames[idx]


def clip_to_tensor(clip: np.ndarray) -> torch.Tensor:
    """(frames, H, W) uint8 -> (frames, 3, 224, 224) normalised for the ViT."""
    t = torch.from_numpy(clip.astype(np.float32)).unsqueeze(1)
    t = F.interpolate(t, size=(VIT_SIZE, VIT_SIZE), mode="bilinear", align_corners=False)
    t = (t / 255.0 - 0.5) / 0.5
    return t.expand(-1, 3, -1, -1).contiguous()


# ---------------------------------------------------------------- MRI
# (CT has its own resampling pipeline in src/imaging/ct.py)
def normalise_volume(volume: np.ndarray, lo_pct: float = 1, hi_pct: float = 99) -> np.ndarray:
    """Percentile clip then scale to [0, 1] -- MRI intensities have no fixed unit."""
    lo, hi = np.percentile(volume, [lo_pct, hi_pct])
    return np.clip((volume.astype(np.float32) - lo) / (hi - lo + 1e-8), 0, 1).astype(np.float32)


def slices_to_tensor(volume01: np.ndarray, size: int = VIT_SIZE) -> torch.Tensor:
    """(H, W, slices) in [0,1] -> (slices, 1, size, size)."""
    t = torch.from_numpy(np.ascontiguousarray(np.moveaxis(volume01, -1, 0))).unsqueeze(1)
    return F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)


@torch.no_grad()
def segment_mri_volume(model: torch.nn.Module, volume: np.ndarray, batch: int = 8) -> np.ndarray:
    """Short-axis MRI volume (H, W, slices) -> label mask of the same shape."""
    h, w, _ = volume.shape
    x = slices_to_tensor(normalise_volume(volume))
    labels = []
    for i in range(0, len(x), batch):
        logits = model(x[i : i + batch])
        logits = F.interpolate(logits, size=(h, w), mode="bilinear", align_corners=False)
        labels.append(logits.argmax(1))
    return np.moveaxis(torch.cat(labels).numpy().astype(np.uint8), 0, -1)


def mask_slices_to_tensor(mask: np.ndarray, size: int = VIT_SIZE) -> torch.Tensor:
    """(H, W, slices) integer labels -> (slices, size, size), nearest-neighbour so labels stay whole."""
    t = torch.from_numpy(np.ascontiguousarray(np.moveaxis(mask, -1, 0))).unsqueeze(1).float()
    return F.interpolate(t, size=(size, size), mode="nearest")[:, 0].long()

## Unpack from Drive

In [ ]:
import glob, os
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/cordia/EchoNet-Dynamic.zip'
if not glob.glob('/content/echonet/**/FileList.csv', recursive=True):
    !unzip -q "$ZIP" -d /content/echonet
root = os.path.dirname(glob.glob('/content/echonet/**/FileList.csv', recursive=True)[0])
print('EchoNet root:', root)

In [ ]:
import numpy as np
import pandas as pd
files = pd.read_csv(f'{root}/FileList.csv')
files['FileName'] = files['FileName'].astype(str).apply(lambda f: f if f.endswith('.avi') else f + '.avi')
files['path'] = files['FileName'].apply(lambda f: f'{root}/Videos/{f}')
files = files[files['path'].apply(os.path.exists)]
files['Split'] = files['Split'].str.upper()
print(files['Split'].value_counts())
print(files['EF'].describe())

## Dataset: a random 16-frame window per training clip, the centred window for val/test

In [ ]:
from torch.utils.data import Dataset, DataLoader
from src.imaging.preprocess import read_video_frames, sample_clip, clip_to_tensor
from src.models.imaging.echo_vit import EchoViT, N_FRAMES

STRIDE = 2

class EchoDataset(Dataset):
    def __init__(self, frame, train):
        self.paths, self.ef, self.train = frame['path'].tolist(), frame['EF'].values.astype(np.float32), train
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        frames = read_video_frames(self.paths[i])
        start = np.random.randint(0, max(1, len(frames) - N_FRAMES * STRIDE)) if self.train else None
        return clip_to_tensor(sample_clip(frames, N_FRAMES, STRIDE, start)), self.ef[i]

split = lambda s: files[files['Split'] == s]
train_loader = DataLoader(EchoDataset(split('TRAIN'), True), batch_size=8, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(EchoDataset(split('VAL'), False), batch_size=8, num_workers=4)
test_loader = DataLoader(EchoDataset(split('TEST'), False), batch_size=8, num_workers=4)

model = EchoViT().to(device)
print('Trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## Train (L1 loss on EF; ViT layers below 10 frozen to fit a T4)

In [ ]:
from torch import nn
EPOCHS = 5
optimiser = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-5, weight_decay=1e-4)
scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda')
criterion = nn.L1Loss()

def predict(loader):
    model.eval(); preds, ys = [], []
    with torch.no_grad(), torch.autocast(device, enabled=device == 'cuda'):
        for xb, yb in loader:
            preds.append(model(xb.to(device)).float().cpu().numpy()); ys.append(yb.numpy())
    return np.concatenate(preds), np.concatenate(ys)

best_mae, best_state = 1e9, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    for step, (xb, yb) in enumerate(train_loader):
        with torch.autocast(device, enabled=device == 'cuda'):
            loss = criterion(model(xb.to(device)), yb.to(device))
        optimiser.zero_grad(); scaler.scale(loss).backward(); scaler.step(optimiser); scaler.update()
        if step % 100 == 0:
            print(f'epoch {epoch} step {step}/{len(train_loader)} L1 {loss.item():.2f}')
    p, y = predict(val_loader)
    mae = np.abs(p - y).mean()
    print(f'Epoch {epoch} val MAE {mae:.2f} EF points')
    if mae < best_mae:
        best_mae, best_state = mae, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)

## Test
For reference, the original EchoNet-Dynamic paper (Ouyang et al., *Nature* 2020) reports MAE ≈ 4.1 EF points using segmentation and beat-to-beat averaging; a single-clip model is expected to be a few points worse.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score
p, y = predict(test_loader)
print(f'Test MAE  {mean_absolute_error(y, p):.2f} EF points')
print(f'Test RMSE {np.sqrt(((y - p) ** 2).mean()):.2f}')
print(f'Test R^2  {r2_score(y, p):.3f}')
print(f'AUROC for reduced EF (< 40%, heart failure range): {roc_auc_score(y < 40, -p):.3f}')

In [ ]:
torch.save(best_state, 'echo_vit.pt')
from google.colab import files
files.download('echo_vit.pt')  # put it in cordia_ai/checkpoints/